In [1]:
# 1. Install required dependencies
!pip install fastapi uvicorn nest_asyncio pydantic scikit-learn xgboost pandas numpy pyngrok -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import threading
import os
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok

print("🧪 1. Preparing chemical nutrient bounds data for Model 2...")

# Crop profiles with clean English names
crop_profiles_m2 = {
    'Wheat': {'N': (80, 120), 'P': (40, 60), 'K': (30, 50), 'pH': (6.0, 7.0)},
    'Barley': {'N': (60, 90), 'P': (30, 50), 'K': (25, 45), 'pH': (6.2, 7.8)},
    'Soybean': {'N': (20, 40), 'P': (60, 90), 'K': (50, 80), 'pH': (6.0, 7.5)},
    'Corn': {'N': (100, 150), 'P': (45, 70), 'K': (40, 65), 'pH': (5.8, 7.0)},
    'Sunflower': {'N': (50, 80), 'P': (40, 60), 'K': (60, 90), 'pH': (6.5, 7.5)},
    'Tomato': {'N': (70, 110), 'P': (40, 70), 'K': (40, 60), 'pH': (6.0, 6.8)},
    'Potato': {'N': (40, 70), 'P': (40, 60), 'K': (45, 60), 'pH': (5.0, 6.5)}
}

# Generate synthetic dataset
np.random.seed(100)
data_m2 = []
for crop, bounds in crop_profiles_m2.items():
    for _ in range(300):
        n = np.random.uniform(*bounds['N']) + np.random.normal(0, 2)
        p = np.random.uniform(*bounds['P']) + np.random.normal(0, 2)
        k = np.random.uniform(*bounds['K']) + np.random.normal(0, 2)
        ph = np.random.uniform(*bounds['pH']) + np.random.normal(0, 0.1)
        data_m2.append([n, p, k, ph, crop])

df_m2 = pd.DataFrame(data_m2, columns=['N', 'P', 'K', 'pH', 'Crop'])

X_m2 = df_m2[['N', 'P', 'K', 'pH']]
y_m2 = df_m2['Crop']

le_m2 = LabelEncoder()
y_encoded_m2 = le_m2.fit_transform(y_m2)

X_train_m2, X_test_m2, y_train_m2, y_test_m2 = train_test_split(X_m2, y_encoded_m2, test_size=0.2, random_state=42)

# 🤖 2. Train and Evaluate Models
print("\n🤖 2. Training and evaluating models for Model 2...")

rf_model_m2 = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model_m2.fit(X_train_m2, y_train_m2)
rf_acc_m2 = accuracy_score(y_test_m2, rf_model_m2.predict(X_test_m2)) * 100

xgb_model_m2 = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model_m2.fit(X_train_m2, y_train_m2)
xgb_acc_m2 = accuracy_score(y_test_m2, xgb_model_m2.predict(X_test_m2)) * 100

print(f"🌲 Random Forest Accuracy: {rf_acc_m2:.2f}%")
print(f"🚀 XGBoost Accuracy: {xgb_acc_m2:.2f}%")

best_model_m2 = xgb_model_m2 if xgb_acc_m2 >= rf_acc_m2 else rf_model_m2
best_name_m2 = "XGBoost" if xgb_acc_m2 >= rf_acc_m2 else "Random Forest"
print(f"✅ Selected best model for Model 2: {best_name_m2}\n")

# Kill any server running on port 8000
os.system("pkill -f uvicorn")

app = FastAPI(
    title="HAK Model 2 - Chemical Nutrient Analysis API",
    description="API for crop prediction and recommendations based on soil chemical properties"
)

class NutrientInput(BaseModel):
    n: float
    p: float
    k: float
    ph: float

@app.post("/predict-model2")
async def predict_model2_endpoint(input_data: NutrientInput):
    try:
        sample = pd.DataFrame([[input_data.n, input_data.p, input_data.k, input_data.ph]],
                              columns=['N', 'P', 'K', 'pH'])

        pred_idx = best_model_m2.predict(sample)[0]
        probs = best_model_m2.predict_proba(sample)[0]

        recommended_crop = le_m2.inverse_transform([pred_idx])[0]
        confidence = float(probs[pred_idx] * 100)

        sorted_indices = np.argsort(probs)
        unsuitable_indices = sorted_indices[:2]
        unsuitable_crops = le_m2.inverse_transform(unsuitable_indices).tolist()

        return {
            "status": "success",
            "model_used": best_name_m2,
            "inputs": {
                "N": input_data.n,
                "P": input_data.p,
                "K": input_data.k,
                "pH": input_data.ph
            },
            "result": {
                "recommended_crop": str(recommended_crop),
                "confidence_percentage": round(confidence, 2),
                "unsuitable_crops": unsuitable_crops
            }
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

def start_server():
    uvicorn.run(app, host="127.0.0.1", port=8000)

server_thread = threading.Thread(target=start_server)
server_thread.daemon = True
server_thread.start()

print("⚡ Model 2 API server running in background!")

# 3. Connect server via ngrok with token
ngrok.set_auth_token("3IeIzONpxNCkRW37WVVCeD8RT5q_Gei9Ro3jU5zNVsRbKW2X")

public_url = ngrok.connect(8000)
print("🔗 Public Static API URL (Model 2):", public_url)

🧪 1. Preparing chemical nutrient bounds data for Model 2...

🤖 2. Training and evaluating models for Model 2...
🌲 Random Forest Accuracy: 88.81%
🚀 XGBoost Accuracy: 88.10%
✅ Selected best model for Model 2: Random Forest

⚡ Model 2 API server running in background!


INFO:     Started server process [4904]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


🔗 Public Static API URL (Model 2): NgrokTunnel: "https://gondola-movable-refuse.ngrok-free.dev" -> "http://localhost:8000"
